# ArmorVault — PP-OCRv5 + Rules + Geometry + E5

A document-field extraction feasibility test using a synthetic Saudi ID. The pipeline is deterministic first: OCR, RTL correction, aliases, data types, geometry, and strict validation. `multilingual-e5-small` only ranks unresolved labels; it never generates field values.

No Google Drive, private image, Qwen, or MiniCPM is used.

In [ ]:
%pip install -q "rapidocr>=3.9,<4" "onnxruntime>=1.20" "huggingface_hub>=0.32,<1" "pyyaml>=6" "transformers==4.51.3" "tokenizers==0.21.4" "python-bidi>=0.6,<1" pillow


In [ ]:
import gc, json, math, re, time, unicodedata
from datetime import datetime
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
import torch, torch.nn.functional as F, yaml
from bidi.algorithm import get_display
from huggingface_hub import hf_hub_download
from PIL import Image
from IPython.display import display

root = Path('/content/armorvault-rules-e5')
root.mkdir(parents=True, exist_ok=True)
image_path = root / 'saudi_id_demo.png'
urlretrieve('https://raw.githubusercontent.com/almawti/armorvault-ocr-vl-gpu-lab/main/saudi_id_demo.png', image_path)
display(Image.open(image_path))
print({'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', 'image': str(image_path)})


In [ ]:
from rapidocr import RapidOCR

det_model = hf_hub_download('PaddlePaddle/PP-OCRv5_mobile_det_onnx', 'inference.onnx')
rec_model = hf_hub_download('PaddlePaddle/arabic_PP-OCRv5_mobile_rec_onnx', 'inference.onnx')
rec_config = hf_hub_download('PaddlePaddle/arabic_PP-OCRv5_mobile_rec_onnx', 'inference.yml')
characters = yaml.safe_load(Path(rec_config).read_text(encoding='utf-8'))['PostProcess']['character_dict']
keys_path = root / 'ppocrv5_arabic_dict.txt'
keys_path.write_text('\n'.join(str(char) for char in characters), encoding='utf-8')
ocr = RapidOCR(params={
    'Det.model_path': det_model,
    'Rec.model_path': rec_model,
    'Rec.rec_keys_path': str(keys_path),
    'Rec.rec_img_shape': [3, 48, 320],
})
started = time.perf_counter()
ocr_result = ocr(str(image_path), use_cls=False)
ocr_seconds = time.perf_counter() - started
texts = list(ocr_result.txts or [])
scores = [float(value) for value in (ocr_result.scores or [])]
boxes = ocr_result.boxes.tolist() if ocr_result.boxes is not None else []
raw_lines = [
    {'id': index, 'text': text, 'confidence': round(scores[index], 4), 'box': boxes[index]}
    for index, text in enumerate(texts)
]
print(json.dumps({'ocrSeconds': round(ocr_seconds, 3), 'rawLines': raw_lines}, ensure_ascii=False, indent=2))
del ocr, ocr_result
_released = gc.collect()


In [ ]:
ARABIC_DIGITS = str.maketrans('٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹', '01234567890123456789')
ARABIC_RE = re.compile(r'[\u0600-\u06ff]')
DATE_RE = re.compile(r'(?<!\d)(\d{1,4})[\s./-]+(\d{1,2})[\s./-]+(\d{1,4})(?!\d)')
NUMBER_RE = re.compile(r'(?<!\d)(\d{6,14})(?!\d)')

def logical_arabic(text):
    # The custom Arabic recognizer emits visual-order runs through RapidOCR's default decoder.
    # Applying the bidi display transform restores logical Arabic while preserving digit runs.
    return get_display(text) if len(ARABIC_RE.findall(text)) >= 2 else text

def normalize(text):
    # Input must already be in logical order. Do not apply bidi conversion twice.
    text = text.translate(ARABIC_DIGITS).lower()
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'[\u064b-\u065f\u0670ـ]', '', text)
    text = text.replace('أ', 'ا').replace('إ', 'ا').replace('آ', 'ا').replace('ى', 'ي')
    return re.sub(r'[^\w\u0600-\u06ff]+', ' ', text).strip()

def center(box):
    points = np.asarray(box, dtype=float)
    return [float(points[:, 0].mean()), float(points[:, 1].mean())]

lines = []
for line in raw_lines:
    corrected = logical_arabic(line['text'])
    lines.append({**line, 'rawText': line['text'], 'text': corrected, 'normalized': normalize(corrected), 'center': center(line['box'])})
print(json.dumps({'rtlCorrectedLines': lines}, ensure_ascii=False, indent=2))


In [ ]:
FIELD_SCHEMA = {
    'documentNumber': {'type': 'number', 'aliases': ['document number', 'identity number', 'id number', 'card number', 'no', 'number', 'رقم الهوية', 'رقم الوثيقة', 'رقم البطاقة', 'الرقم']},
    'birthDate': {'type': 'date', 'aliases': ['date of birth', 'birth date', 'dob', 'تاريخ الميلاد', 'تاريخ الولادة']},
    'issueDate': {'type': 'date', 'aliases': ['issue date', 'date of issue', 'doi', 'issued on', 'تاريخ الاصدار', 'تاريخ الإصدار']},
    'expiryDate': {'type': 'date', 'aliases': ['expiry date', 'expiration date', 'date of expiry', 'valid until', 'doe', 'تاريخ الانتهاء', 'تاريخ انتهاء الصلاحية']},
    'holderNameArabic': {'type': 'arabic_name', 'aliases': ['holder name', 'full name', 'name', 'اسم حامل الوثيقة', 'الاسم الكامل', 'الاسم']},
    'holderNameEnglish': {'type': 'latin_name', 'aliases': ['holder name', 'full name', 'name', 'name in english', 'الاسم بالانجليزية']},
}
NORMALIZED_ALIASES = {field: [normalize(alias) for alias in spec['aliases']] for field, spec in FIELD_SCHEMA.items()}

def parse_date(text):
    match = DATE_RE.search(text.translate(ARABIC_DIGITS))
    if not match:
        return None
    a, b, c = [int(value) for value in match.groups()]
    candidates = [(c, b, a), (a, b, c)] if a <= 31 else [(a, b, c)]
    for year, month, day in candidates:
        if year < 100:
            year += 2000 if year < 50 else 1900
        try:
            return datetime(year, month, day).strftime('%Y-%m-%d')
        except ValueError:
            pass
    return None

def alias_match(line, field):
    normalized = line['normalized']
    return max((len(alias) for alias in NORMALIZED_ALIASES[field] if re.search(r'(^|\s)' + re.escape(alias) + r'(\s|$)', normalized)), default=0)

def compatible_values(field):
    kind = FIELD_SCHEMA[field]['type']
    candidates = []
    for line in lines:
        text = line['text'].translate(ARABIC_DIGITS)
        value = None
        if kind == 'date':
            value = parse_date(text)
        elif kind == 'number':
            matches = NUMBER_RE.findall(text)
            value = max(matches, key=len) if matches else None
        if value:
            candidates.append((line, value))
    return candidates

def geometry_distance(label, value):
    dx = abs(value['center'][0] - label['center'][0])
    dy = abs(value['center'][1] - label['center'][1])
    same_row_bonus = 0.45 if dy <= 25 else 1.0
    return math.hypot(dx, dy * 2.2) * same_row_bonus

resolved, evidence, rule_trace = {}, {}, []
for field in ('documentNumber', 'birthDate', 'issueDate', 'expiryDate'):
    labels = [line for line in lines if alias_match(line, field)]
    values = compatible_values(field)
    ranked = []
    for label in labels:
        for value_line, value in values:
            inline = label['id'] == value_line['id']
            distance = 0.0 if inline else geometry_distance(label, value_line)
            ranked.append((not inline, distance, -alias_match(label, field), label, value_line, value))
    if ranked:
        _, distance, _, label, value_line, value = sorted(ranked, key=lambda item: item[:3])[0]
        resolved[field] = value
        evidence[field] = sorted(set([label['id'], value_line['id']]))
        rule_trace.append({'field': field, 'decision': value, 'label': label['text'], 'source': value_line['text'], 'distance': round(distance, 2)})
    else:
        resolved[field] = None
        rule_trace.append({'field': field, 'decision': None, 'reason': 'no compatible labeled value'})

header_text = ' '.join(line['normalized'] for line in lines[:6])
resolved['documentType'] = 'saudi_national_id' if ('الهوية الوطنية' in header_text or 'المملكة العربية السعودية' in header_text) else 'other'

latin_names = [line for line in lines if re.fullmatch(r'[A-Z][A-Z, .-]{8,}', line['text']) and len(line['text'].replace(',', ' ').split()) >= 3]
if latin_names:
    chosen = max(latin_names, key=lambda line: line['confidence'])
    resolved['holderNameEnglish'], evidence['holderNameEnglish'] = chosen['text'], [chosen['id']]
    latin_name_line = chosen
else:
    resolved['holderNameEnglish'] = None
    latin_name_line = None

arabic_exclusion_phrases = [normalize(value) for value in (
    'الهوية الوطنية', 'المملكة العربية السعودية', 'وزارة الداخلية',
    'اسم حامل الوثيقة', 'الاسم الكامل', 'تاريخ الميلاد', 'تاريخ الانتهاء',
    'مكان الميلاد', 'رقم الهوية', 'رقم الوثيقة',
)]

def arabic_name_score(line):
    normalized = line['normalized']
    words = normalized.split()
    if len(words) < 3 or sum(bool(ARABIC_RE.search(word)) for word in words) < 3:
        return None
    if any(phrase in normalized for phrase in arabic_exclusion_phrases):
        return None
    score = float(line['confidence'])
    if normalize('بنت') in words:
        score += 3.0
    elif normalize('بن') in words:
        score += 2.0
    if latin_name_line is not None:
        vertical_distance = abs(line['center'][1] - latin_name_line['center'][1])
        score += max(0.0, 1.5 - vertical_distance / 80.0)
    return score

arabic_names = [(score, line) for line in lines if (score := arabic_name_score(line)) is not None]
if arabic_names:
    _, chosen = max(arabic_names, key=lambda item: item[0])
    resolved['holderNameArabic'], evidence['holderNameArabic'] = chosen['text'], [chosen['id']]
    rule_trace.append({'field': 'holderNameArabic', 'decision': chosen['text'], 'source': chosen['text'], 'score': round(arabic_name_score(chosen), 3)})
else:
    resolved['holderNameArabic'] = None
print(json.dumps({'deterministicResult': resolved, 'evidence': evidence, 'trace': rule_trace}, ensure_ascii=False, indent=2))


In [ ]:
from transformers import AutoModel, AutoTokenizer

e5_name = 'intfloat/multilingual-e5-small'
started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(e5_name)
e5 = AutoModel.from_pretrained(e5_name).eval()
e5_init_seconds = time.perf_counter() - started

def embed(texts, prefix):
    encoded = tokenizer([prefix + text for text in texts], padding=True, truncation=True, max_length=128, return_tensors='pt')
    with torch.inference_mode():
        output = e5(**encoded).last_hidden_state
    mask = encoded['attention_mask'].unsqueeze(-1).expand(output.size()).float()
    pooled = (output * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return F.normalize(pooled, p=2, dim=1)

field_descriptions = {
    field: field + ': ' + ', '.join(spec['aliases']) + '; expected value type: ' + spec['type']
    for field, spec in FIELD_SCHEMA.items()
}
candidate_lines = [line for line in lines if not DATE_RE.search(line['text'].translate(ARABIC_DIGITS)) and not re.fullmatch(r'[\d\s]+', line['text'].translate(ARABIC_DIGITS))]
started = time.perf_counter()
query_vectors = embed([line['text'] for line in candidate_lines], 'query: ')
field_names = list(field_descriptions)
field_vectors = embed([field_descriptions[field] for field in field_names], 'passage: ')
similarities = query_vectors @ field_vectors.T
e5_seconds = time.perf_counter() - started
suggestions = []
for index, line in enumerate(candidate_lines):
    ranked = torch.argsort(similarities[index], descending=True).tolist()
    best, second = ranked[0], ranked[1]
    suggestions.append({
        'lineId': line['id'],
        'text': line['text'],
        'bestField': field_names[best],
        'bestScore': round(float(similarities[index, best]), 4),
        'secondField': field_names[second],
        'secondScore': round(float(similarities[index, second]), 4),
        'margin': round(float(similarities[index, best] - similarities[index, second]), 4),
        'acceptedAutomatically': False,
    })
print(json.dumps({'e5InitializationSeconds': round(e5_init_seconds, 3), 'e5InferenceSeconds': round(e5_seconds, 3), 'note': 'Scores are rankings, not calibrated probabilities. This benchmark does not auto-accept E5 suggestions.', 'suggestions': suggestions}, ensure_ascii=False, indent=2))


In [ ]:
expected = {
    'documentType': 'saudi_national_id',
    'documentNumber': '123456789',
    'holderNameArabic': 'نورة بنت عبدالرحمن أحمد',
    'holderNameEnglish': 'NOURAH, ABDULRAHMAN AHMAD',
    'birthDate': '1992-04-28',
    'issueDate': None,
    'expiryDate': '2032-01-19',
}
comparison = {}
for field, expected_value in expected.items():
    actual = resolved.get(field)
    comparison[field] = {'actual': actual, 'expected': expected_value, 'exactMatch': normalize(str(actual or '')) == normalize(str(expected_value or ''))}
exact = sum(item['exactMatch'] for item in comparison.values())
final_result = {
    'models': {'ocr': 'PP-OCRv5 Mobile detector + Arabic recognizer ONNX', 'semanticMatcher': e5_name},
    'timings': {'ocrSeconds': round(ocr_seconds, 3), 'e5InitializationSeconds': round(e5_init_seconds, 3), 'e5InferenceSeconds': round(e5_seconds, 3)},
    'deterministicResult': resolved,
    'evidence': evidence,
    'ruleTrace': rule_trace,
    'e5SuggestionsNotAutoAccepted': suggestions,
    'comparison': comparison,
    'exactFields': exact,
    'totalFields': len(comparison),
}
(root / 'result.json').write_text(json.dumps(final_result, ensure_ascii=False, indent=2), encoding='utf-8')
print('\nFINAL RESULT\n')
print(json.dumps(final_result, ensure_ascii=False, indent=2))


The final result is printed above and saved temporarily at `/content/armorvault-rules-e5/result.json`. Do not interpret E5 cosine scores as confidence percentages; thresholds require calibration on a larger document benchmark.